## 1. Data Loading

This section imports the required libraries, defines the dataset path in Kaggle, and loads the main dataset files.
It also reads the CSV file and checks the number of images, masks, and records to make sure the dataset is correctly connected and ready for cleaning.

In [ ]:
# Import the required libraries for data loading, cleaning,  preprocessing, and training
import os                          # Used to manage files and folder paths
import cv2                         # Used to read and process images
import pandas as pd                # Used to read and inspect the CSV file
import matplotlib.pyplot as plt    # Used to display images
import xml.etree.ElementTree as ET # Used to read XML annotation files
from PIL import Image              # Used to check if images are valid or corrupted
import shutil                      # Used to copy images and label files
from sklearn.model_selection import train_test_split  # Used to split data into training and validation sets
!pip install ultralytics
from ultralytics import YOLO

In [ ]:
# Define the dataset path in Kaggle and check the main dataset files

dataset_path = "/kaggle/input/datasets/trainingdatapro/parking-space-detection-dataset"
print("Dataset files:", os.listdir(dataset_path))

In [ ]:
# Define the main file and folder locations inside the dataset

images_path = dataset_path + "/images"          # Folder containing original parking images
boxes_path = dataset_path + "/boxes"            # Folder containing mask/box images
csv_path = dataset_path + "/parking.csv"        # CSV file containing image and mask information
xml_path = dataset_path + "/annotations.xml"    # XML file containing parking space annotations

In [ ]:
# Read the CSV file and display the first five rows to inspect the dataset records

df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Check the number of images, masks, and CSV records

print("Number of images:", len(os.listdir(images_path)))
print("Number of masks:", len(os.listdir(boxes_path)))
print("CSV rows:", len(df))

## 2. Data Cleaning
This section checks all parking images to make sure they are readable and not corrupted before using them for model training.

In [ ]:
# Check all images to make sure they are readable and not corrupted before preprocessing

valid_images = []
corrupted_images = []

# Loop through each image and verify that it can be opened correctly
for img_name in os.listdir(images_path):
    img_path = os.path.join(images_path, img_name)

    try:
        img = Image.open(img_path)
        img.verify()
        valid_images.append(img_name)
    except Exception:
        corrupted_images.append(img_name)

# Print the cleaning results
print("Total images:", len(os.listdir(images_path)))
print("Valid images:", len(valid_images))
print("Corrupted images:", len(corrupted_images))
print("Corrupted files:", corrupted_images)

## 3. Data Preprocessing

This section prepares the dataset for YOLO training. It reads the XML annotation file, converts polygon annotations into YOLO TXT label format, splits the images and labels into training and validation sets, and creates the data.yaml configuration file required by YOLO.

In [ ]:
# Read the XML annotation file that contains parking space labels and coordinates

tree = ET.parse(xml_path)
root = tree.getroot()

# Print basic information about the XML file
print("Root tag:", root.tag)
print("Number of annotated images:", len(root.findall("image")))

# Display the attributes of the first annotated image to understand the XML structure
first_image = root.findall("image")[0]
print("First image attributes:", first_image.attrib)

In [ ]:
# Convert XML polygon annotations into YOLO label format

output_labels_path = "/kaggle/working/yolo_labels"
os.makedirs(output_labels_path, exist_ok=True)

# Define class IDs for the two parking space classes
class_map = {
    "free_parking_space": 0,
    "not_free_parking_space": 1
}

# Loop through each image in the XML file and convert its annotations into YOLO labels
for image_tag in root.findall("image"):
    image_name = image_tag.attrib["name"]
    image_width = float(image_tag.attrib["width"])
    image_height = float(image_tag.attrib["height"])

    label_lines = []

    # Loop through each polygon annotation in the current image
    for polygon in image_tag.findall("polygon"):
        label = polygon.attrib["label"]

        if label not in class_map:
            continue

        class_id = class_map[label]
        points = polygon.attrib["points"].split(";")

        x_points = []
        y_points = []

        # Extract all x and y coordinates from the polygon points
        for point in points:
            x, y = point.split(",")
            x_points.append(float(x))
            y_points.append(float(y))

        # Convert polygon coordinates into a rectangular bounding box
        x_min = min(x_points)
        y_min = min(y_points)
        x_max = max(x_points)
        y_max = max(y_points)

        # Normalize bounding box values to match YOLO format
        x_center = ((x_min + x_max) / 2) / image_width
        y_center = ((y_min + y_max) / 2) / image_height
        box_width = (x_max - x_min) / image_width
        box_height = (y_max - y_min) / image_height

        # Save the annotation line in YOLO format: class x_center y_center width height
        label_lines.append(
            f"{class_id} {x_center} {y_center} {box_width} {box_height}"
        )

    # Create a TXT label file with the same name as the image
    txt_name = os.path.splitext(os.path.basename(image_name))[0] + ".txt"
    txt_path = os.path.join(output_labels_path, txt_name)

    # Write all YOLO labels for the current image into the TXT file
    with open(txt_path, "w") as f:
        f.write("\n".join(label_lines))

print("YOLO labels created successfully!")
print("Number of label files:", len(os.listdir(output_labels_path)))

In [ ]:
# Split images and labels into training and validation sets

yolo_dataset_path = "/kaggle/working/parking_yolo_dataset"

folders = [
    "images/train", "images/val",
    "labels/train", "labels/val"
]

# Create YOLO folder structure
for folder in folders:
    os.makedirs(os.path.join(yolo_dataset_path, folder), exist_ok=True)

# Split image files into 80% training and 20% validation
image_files = sorted(os.listdir(images_path))

train_files, val_files = train_test_split(
    image_files,
    test_size=0.2,
    random_state=42
)

# Copy images and their matching labels to the correct folder
def copy_files(file_list, split_name):
    for img_name in file_list:
        label_name = os.path.splitext(img_name)[0] + ".txt"

        shutil.copy(
            os.path.join(images_path, img_name),
            os.path.join(yolo_dataset_path, "images", split_name, img_name)
        )

        shutil.copy(
            os.path.join(output_labels_path, label_name),
            os.path.join(yolo_dataset_path, "labels", split_name, label_name)
        )

copy_files(train_files, "train")
copy_files(val_files, "val")

print("Training images:", len(train_files))
print("Validation images:", len(val_files))


In [ ]:
# Create a data.yaml file that tells YOLO where the training and validation images are located,
# and defines the number and names of the parking space classes.

yaml_content = """
train: /kaggle/working/parking_yolo_dataset/images/train
val: /kaggle/working/parking_yolo_dataset/images/val

nc: 2
names:
  0: free_parking_space
  1: not_free_parking_space
"""

# Save the data.yaml file inside the prepared YOLO dataset folder
yaml_path = "/kaggle/working/parking_yolo_dataset/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")
print("YAML file path:", yaml_path)

In [ ]:
# Verify that the YOLO dataset folders were created correctly

print("Main YOLO dataset folders:", os.listdir("/kaggle/working/parking_yolo_dataset"))
print("Image folders:", os.listdir("/kaggle/working/parking_yolo_dataset/images"))
print("Label folders:", os.listdir("/kaggle/working/parking_yolo_dataset/labels"))

## 4. Model Training (Baseline Fine-Tuning)

In [ ]:
# Train YOLOv8 nano model for 20 epochs as a baseline experiment

model_20 = YOLO("yolov8n.pt")

results_20 = model_20.train(
    data="/kaggle/working/parking_yolo_dataset/data.yaml",
    epochs=20,      # Baseline training rounds
    imgsz=640,      # Resize input images to 640x640
    batch=8,        # Number of images processed together
    name="parking_baseline_yolov8n_20epochs"
)

In [ ]:
# Load a pre-trained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Train the YOLO model on our parking dataset
results = model.train(
    # Path to the YAML file that tells YOLO where the training and validation images are
    data="/kaggle/working/parking_yolo_dataset/data.yaml",
    epochs=50, # Number of training rounds
    imgsz=640, # Resize all input images to 640x640 during training
    batch=8,# Number of images processed together in one training step
    # YOLO will save the results under this folder name
    name="parking_baseline_yolov8n_50epochs" 
)


## 5. Model Evaluation 

In [ ]:
# Evaluate the 20-epoch YOLO model on the validation dataset
model_20_eval = YOLO("/kaggle/working/runs/detect/parking_baseline_yolov8n_20epochs/weights/best.pt")

metrics_20 = model_20_eval.val(
    data="/kaggle/working/parking_yolo_dataset/data.yaml"
)

# Print the evaluation scores for the 20-epoch model
print("20 Epochs Results")
print("Precision:", metrics_20.box.mp)      # Precision measures how many detected parking spaces were correctly predicted
print("Recall:", metrics_20.box.mr)         # Recall measures how many real parking spaces were successfully detected
print("mAP50:", metrics_20.box.map50)       # Measures overall object detection accuracy at IoU 0.50
print("mAP50-95:", metrics_20.box.map)      # Measures detection performance with stricter evaluation

In [ ]:
# Evaluate the trained YOLO model on the validation dataset
metrics = model.val()

# Print the Precision score
print("Precision:", metrics.box.mp)# Precision measures how many detected parking spaces were predicted correctly
print("Recall:", metrics.box.mr)# Recall measures how many real parking spaces were successfully detected by the model
print("mAP50:", metrics.box.map50)# Measures overall object detection accuracy
print("mAP50-95:", metrics.box.map)# Measures detection performance with stricter evaluation

In [ ]:
# Create a comparison table between the 20-epoch and 50-epoch models

comparison_table = {
    "Model": ["YOLOv8n Baseline", "YOLOv8n Improved"],
    "Epochs": [20, 50],
    "Precision": [metrics_20.box.mp, metrics.box.mp],
    "Recall": [metrics_20.box.mr, metrics.box.mr],
    "mAP50": [metrics_20.box.map50, metrics.box.map50],
    "mAP50-95": [metrics_20.box.map, metrics.box.map]
}

df_comparison = pd.DataFrame(comparison_table)
df_comparison

## 6. Training Results Visualization

In [ ]:
# Path to the training results image generated by YOLO
results_path = "/kaggle/working/runs/detect/parking_baseline_yolov8n_50epochs/results.png"

results_img = cv2.imread(results_path)# Read the results image
# Convert image colors from BGR to RGB for correct display
results_img = cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB)

# Display the training results image
plt.figure(figsize=(15,10))
plt.imshow(results_img)
plt.axis("off")
plt.show()

## 7.Prediction & Testing

In [ ]:
# "best.pt" contains the weights of the model that achieved the best performance during training
model = YOLO("/kaggle/working/runs/detect/parking_baseline_yolov8n_50epochs/weights/best.pt")

# Select one image from the validation dataset for testing
# Here we selected the third image using index [2]
image_path = "/kaggle/working/parking_yolo_dataset/images/val/" + os.listdir("/kaggle/working/parking_yolo_dataset/images/val")[2]

# Prediction
results = model.predict(source=image_path, conf=0.25)# conf=0.25 means the model will display detections with confidence greater than 25%

# Draw the prediction results
result_img = results[0].plot()
# Convert image colors from BGR to RGB for correct display in Matplotlib
result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

# Display the predicted image
plt.figure(figsize=(15,10))
plt.imshow(result_img)
plt.axis("off")
plt.show()